In [5]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import datetime
from pathlib import Path

import dotenv

dotenv.load_dotenv("tokens.env")

curr_date: str = datetime.datetime.now().isoformat().split("T")[0]

print(curr_date)

/mnt/sdd1/atharvas/formulacode/datasmith
2025-09-20


In [ ]:
import docker
from tqdm.auto import tqdm

all_images = [
    line.split()[10]
    for line in Path("scratch/scripts/parallel_validate_containers.log").read_text().splitlines()
    if "buildx build" in line
]
images_to_remove = [
    line.split()[3]
    for line in Path("scratch/scripts/parallel_validate_containers.log").read_text().splitlines()
    if "failed to run" in line
]
images_to_keep = list(set(all_images) - set(images_to_remove))
print(f"Keeping {len(images_to_keep)} images, removing {len(images_to_remove)} images")
# Increase timeout significantly for large image operations
client = docker.from_env(timeout=3600)  # 1 hour timeout
for img in tqdm(images_to_remove):
    try:
        client.images.remove(img, force=True)
    except Exception as e:
        print(f"Failed to remove image {img}: {e}")
        continue

Keeping 325 images, removing 92 images


  0%|          | 0/92 [00:00<?, ?it/s]

100%|██████████| 92/92 [00:00<00:00, 535.00it/s]


In [3]:
from docker.models.images import Image


def get_image(image_name: str) -> Image | None:
    client = docker.from_env(timeout=3600)  # 1 hour timeout
    try:
        return client.images.get(image_name)
    except Exception:
        return None


images = {img_name.split(":")[0]: get_image(img_name) for img_name in images_to_keep}
images = {img_name: img for img_name, img in images.items() if img is not None}
print(f"Found {len(images)} images locally")

Found 80 images locally


In [ ]:
import contextlib
import gzip
import json
import os
import re
import shutil

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config as BotoConfig


def _safe_tar_name(image_name: str) -> str:
    """
    Make a filesystem- and S3-friendly tar name from the image name.
    e.g. 'registry:5000/ns/app:1.2.3' -> 'registry_5000_ns_app_1.2.3.tar.gz'
    """
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", image_name.replace("/", "_").replace(":", "_"))
    return f"{base}.tar.gz"


def _save_image_to_tar_gz(img, tar_gz_path: str, max_retries: int = 3) -> None:
    """
    Stream-save a docker image to tar.gz without loading it all into memory.
    Includes retry logic for timeout issues.
    """
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(tar_gz_path), exist_ok=True)

    temp_tar_path = tar_gz_path.replace(".tar.gz", ".tar")

    for attempt in range(max_retries):
        try:
            print(f"[+] Attempt {attempt + 1}/{max_retries}: Saving image to {temp_tar_path}")

            # image.save(named=True) yields a generator of bytes
            stream = img.save(named=True)

            # Write to a temporary tar file first, then compress
            with open(temp_tar_path, "wb") as f:
                for chunk in stream:
                    f.write(chunk)

            print(f"[+] Compressing {temp_tar_path} to {tar_gz_path}")
            # Compress the tar file to tar.gz
            with open(temp_tar_path, "rb") as f_in, gzip.open(tar_gz_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

            # Remove the temporary tar file
            os.remove(temp_tar_path)
            print("[✓] Successfully saved and compressed image")
        except Exception as e:
            print(f"[!] Attempt {attempt + 1} failed: {e}")
            # Clean up any partial files
            for path in [temp_tar_path, tar_gz_path]:
                if os.path.exists(path):
                    with contextlib.suppress(Exception):
                        os.remove(path)

            if attempt == max_retries - 1:
                raise
            else:
                print("[+] Retrying in 5 seconds...")
                import time

                time.sleep(5)
        else:
            return


def upload_images_to_s3(
    images: dict[str, "Image"],
    bucket: str,
    prefix: str = "",
    region: str | None = None,
    sse: str | None = None,  # e.g. "AES256" or "aws:kms"
    tarball_dir: str = "scratch/artifacts/tarballs",
    force: bool = False,
) -> dict[str, str]:
    """
    For each {name: Image} entry, produce a .tar.gz and upload to s3://bucket/prefix/<name>.tar.gz
    Also saves the tarballs locally to tarball_dir.
    If force is enabled, the local tarball and uploaded tarball are overwritten.
    """
    session = boto3.session.Session(region_name=region)
    s3 = session.client("s3", config=BotoConfig(retries={"max_attempts": 10}))
    tx_config = TransferConfig(multipart_threshold=8 * 1024 * 1024, multipart_chunksize=8 * 1024 * 1024)

    mapping: dict[str, str] = {}

    for name, img in images.items():
        tar_gz_name = _safe_tar_name(name)
        key = f"{prefix.strip('/')}/{tar_gz_name}" if prefix else tar_gz_name

        # Local path for the tarball
        local_tar_gz_path = os.path.join(tarball_dir, tar_gz_name)
        if os.path.exists(local_tar_gz_path) and not force:
            print(f"[+] Skipping {name} as it already exists locally")
            continue

        try:
            print(f"[+] Saving image '{name}' → {local_tar_gz_path}")
            _save_image_to_tar_gz(img, local_tar_gz_path)

            extra = {"ContentType": "application/gzip"}
            if sse:
                extra["ServerSideEncryption"] = sse

            print(f"[+] Uploading {local_tar_gz_path} → s3://{bucket}/{key}")
            s3.upload_file(local_tar_gz_path, bucket, key, ExtraArgs=extra, Config=tx_config)

            uri = f"s3://{bucket}/{key}"
            mapping[name] = uri
            print(f"[✓] Uploaded: {uri}")
        except Exception as e:
            print(f"[!] Error processing {name}: {e}")
            # Clean up the local file if it was created but upload failed
            if os.path.exists(local_tar_gz_path):
                with contextlib.suppress(Exception):
                    os.remove(local_tar_gz_path)

    return mapping


mapping = upload_images_to_s3(
    images,
    bucket=os.environ["AWS_S3_BUCKET"],
    prefix="docker-tarballs/",
    region=os.environ.get("AWS_REGION", None),
)

Path("Scratch/artifacts/aws_docker_image_s3_mapping.json").write_text(json.dumps(mapping, indent=2))

[+] Saving image 'textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325' → scratch/artifacts/tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar.gz
[+] Attempt 1/3: Saving image to scratch/artifacts/tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar
[+] Compressing scratch/artifacts/tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar to scratch/artifacts/tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar.gz
[✓] Successfully saved and compressed image
[+] Uploading scratch/artifacts/tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar.gz → s3://test-datasmith-bucket/docker-tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar.gz
[✓] Uploaded: s3://test-datasmith-bucket/docker-tarballs/textualize-rich-577def22dd83b999c5937cb2bf2ed49600943325.tar.gz
[+] Saving image 'dwavesystems-dimod-9f0b673014c6fb92894f167cd4e88a02817ef1c3' → scratch/artifacts/tarballs/dwavesystems-dimod-9f0b67

KeyboardInterrupt: 

In [ ]:
# #!/usr/bin/env bash
# set -euo pipefail

# S3_URI="s3://my-artifacts-bucket/bundles/docker-images-2025-09-19.tar.gz"
# WORKDIR="${WORKDIR:-/tmp/docker-image-bundle}"

# mkdir -p "$WORKDIR"
# cd "$WORKDIR"

# echo "Downloading bundle..."
# aws s3 cp "$S3_URI" ./bundle.tar.gz

# echo "Extracting bundle..."
# tar -xzf bundle.tar.gz

# echo "Loading images into Docker..."
# shopt -s nullglob
# for img_tar in *.tar; do
#   echo "Loading $img_tar ..."
#   docker load -i "$img_tar"
# done

# echo "Done. Loaded images:"
# docker images
